<a href="https://colab.research.google.com/github/ctr/medical-physics-demos-2026-03/blob/main/MedPhys_B1plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive demo of $B_1^+$ and $B_1^-$ phasor summation (note still some bugs)

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout, Checkbox
import ipywidgets as widgets

# Refactored function to plot a single phasor on a given axes object
def plot_single_phasor_on_ax(Ax, phix_deg, Ay, phiy_deg, t_phase_deg, ax, display_text=True, display_legend=True, display_labels=True):
    # Convert degrees to radians
    phix = np.radians(phix_deg)
    phiy = np.radians(phiy_deg)
    t_phase = np.radians(t_phase_deg)

    # Time vector for the "faint trace" (one full cycle)
    t_full = np.linspace(0, 2*np.pi, 200)
    trace_x = Ax * np.cos(t_full + phix) - Ay * np.sin(t_full + phiy)
    trace_y = Ax * np.sin(t_full + phix) + Ay * np.cos(t_full + phiy)

    # Instantaneous values at the specific moment t_phase
    bx_now = Ax * np.cos(t_phase + phix)
    by_now = Ay * np.cos(t_phase + phix)

    # REF Hoult, The principle of reciprocity, 2000.
    b1plus_re_now = 0.5 * (Ax * np.cos(t_phase + phix) - Ay * np.sin(t_phase + phiy))
    b1plus_im_now = 0.5 * (Ax * np.sin(t_phase + phix) + Ay * np.cos(t_phase + phiy))

    b1minus_re_now = 0.5 * ( Ax * np.cos(t_phase + phix) + Ay * np.sin(t_phase + phiy))
    b1minus_im_now = 0.5 * (-Ax * np.sin(t_phase + phix) + Ay * np.cos(t_phase + phiy))

    # Calculate B1+ (Resultant) amplitude and phase
    res_vec = bx_now + 1j * by_now
    b1_amp = np.abs(res_vec)
    b1_phase = np.degrees(np.angle(res_vec))

    # Setup Plot limits
    # Calculate limit based on the maximum possible resultant amplitude and ensure a minimum
    #dynamic_limit = 1.1 * np.sqrt(Ax**2 + Ay**2)
    #limit = max(dynamic_limit, 1.2) # Ensure a minimum limit of 1.2
    limit = 1.1 * (Ax + Ay)

    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.axhline(0, color='black', lw=1)
    ax.axvline(0, color='black', lw=1)

    # Draw the faint trace (the path the tip follows)
    ax.plot(trace_x, trace_y, color='gray', linestyle=':', alpha=0.5, label='Tip Path')

    # Draw Bx and By components as bold green vectors
    ax.quiver(0, 0, bx_now, 0, angles='xy', scale_units='xy', scale=1,
              color='green', width=0.015, label='$B_x$ Component')
    ax.quiver(0, 0, 0, by_now, angles='xy', scale_units='xy', scale=1,
              color='forestgreen', width=0.015, label='$B_y$ Component')

    # Draw the Resultant B1+ vector
    ax.quiver(0, 0, b1plus_re_now, b1plus_im_now, angles='xy', scale_units='xy', scale=1,
              color='red', width=0.01, label='$B_1^+$ Resultant')

    # Draw the Resultant B1- vector
    ax.quiver(0, 0, b1minus_re_now, b1minus_im_now, angles='xy', scale_units='xy', scale=1,
              color='blue', width=0.01, label='$B_1^+$ Resultant')

    # Draw the B1 + B1- vector end-to-end
    ax.quiver(b1plus_re_now, b1plus_im_now, b1minus_re_now, b1minus_im_now, angles='xy', scale_units='xy', scale=1,
              color='lightskyblue', width=0.01, label='$B_1^+$ Resultant')

    # Draw dashed lines to show the "box" of the addition of Bx and By
    ax.plot([bx_now, bx_now], [0, by_now], 'g--', alpha=0.3)
    ax.plot([0, bx_now], [by_now, by_now], 'g--', alpha=0.3)

    # Draw dashed lines to show the "box" of B1+ + B1- # TODO
    #ax.plot([b1plus_re_now, b1plus_re_now], [0, by_now], 'b--', alpha=0.3)
    #ax.plot([0, bx_now], [by_now, by_now], 'b--', alpha=0.3)

    if display_text:
        text_str = (f"Resultant $B_1^+$:\n"
                    f"Amplitude: {b1_amp:.2f}\n"
                    f"Phase: {b1_phase:.1f}°")
        ax.text(limit*0.5, limit*0.8, text_str, fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='red'))

    if display_labels:
        ax.set_xlabel("X-axis (Gauss or arbitrary)")
        ax.set_ylabel("Y-axis (Gauss or arbitrary)")
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")

    if display_legend:
        ax.legend(loc='lower right')

# New interactive function for display
def interactive_phasor_display(Ax, phix_deg, Ay, phiy_deg, t_phase_deg, show_grid):
    if show_grid:
        fig, axes = plt.subplots(2, 6, figsize=(16, 6)) # 2x6 grid as requested, keeping width
        axes = axes.flatten() # Flatten the 2D array of axes for easier iteration

        phase_increments = np.linspace(0, 360, 12, endpoint=False) # 12 equally spaced phases for a 2x6 grid

        for i, current_t_phase_deg in enumerate(phase_increments):
            ax = axes[i]
            plot_single_phasor_on_ax(Ax, phix_deg, Ay, phiy_deg, current_t_phase_deg, ax,
                                     display_text=False, display_legend=False, display_labels=False) # Simplified for grid
            ax.set_title(f"t={current_t_phase_deg:.0f}°", fontsize=10)

        plt.suptitle(f"Phasor Addition Grid (2x6) (Ax={Ax:.1f}, φx={phix_deg:.0f}°, Ay={Ay:.1f}, φy={phiy_deg:.0f}°)", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(8, 8))
        plot_single_phasor_on_ax(Ax, phix_deg, Ay, phiy_deg, t_phase_deg, ax,
                                     display_text=True, display_legend=True, display_labels=True)
        ax.set_title(f"Phasor Addition: $B_x$ and $B_y$ forming $B_1^+$ (t={t_phase_deg:.0f}°)", fontsize=14)
        plt.show()

# Define Interactive Sliders
style = {'description_width': 'initial'}
layout = Layout(width='500px')

interact(interactive_phasor_display,
         Ax=FloatSlider(value=1.0, min=0, max=2.0, step=0.1, description='Amp Bx', style=style, layout=layout),
         phix_deg=FloatSlider(value=0, min=-180, max=180, step=5, description='Phase Bx (°)', style=style, layout=layout),
         Ay=FloatSlider(value=1.0, min=0, max=2.0, step=0.1, description='Amp By', style=style, layout=layout),
         phiy_deg=FloatSlider(value=0, min=-180, max=180, step=5, description='Phase By (°)', style=style, layout=layout),
         t_phase_deg=FloatSlider(value=0, min=0, max=360, step=5, description='RF Cycle Time (°)', style=style, layout=layout),
         show_grid=Checkbox(value=False, description='Show 2x6 Grid (12 phases)') # Updated checkbox description
);

interactive(children=(FloatSlider(value=1.0, description='Amp Bx', layout=Layout(width='500px'), max=2.0, styl…